<a href="https://colab.research.google.com/github/Rahul-Garg2006/DNN_Project/blob/main/defect_dataset_builder_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D print defect dataset builder (2 datasets -> 1 merged dataset)

**What this notebook does, in simple words**
1. Downloads dataset 1 (Roboflow "Defects Classification in 3D Printing") straight into Colab.
2. Unzips dataset 2 (Bast et al., from Zenodo) after you upload it to Google Drive.
3. Shows you what is inside both, so you can decide the class names.
4. Merges them into one dataset: same class names, no duplicates, split into **train / val / test**.
5. Saves the result as one zip file in your Google Drive, ready for the training notebook.

**How to use it:** run the cells **from top to bottom, one at a time**. Click the play button on the left of a cell and wait for the green tick before running the next one. The cells marked **YOU EDIT THIS** are the only ones where you may need to change something.

You do **not** need a GPU for this notebook.

## Step 1: Connect Google Drive and set the folders
A popup will ask for permission to use your Google Drive. Click **Connect to Google Drive** and then **Allow**.
Two places are used:
- **Google Drive** (`defect_project` folder): permanent. Your Bast zip goes here, and the final merged dataset is saved here.
- **Colab disk** (`/content/work`): fast but temporary. It is wiped when the Colab session ends, so a later session needs the steps re-run (or just use the zip saved in Drive).

In [ ]:
import os, re, csv, shutil, hashlib, zipfile, collections
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab, skipping the Drive mount.')

DRIVE = Path(os.environ.get('DEFECT_DRIVE', '/content/drive/MyDrive/defect_project'))
WORK  = Path(os.environ.get('DEFECT_WORK',  '/content/work'))
RAW   = WORK / 'raw'        # downloaded / unzipped datasets, one folder each
STAGE = WORK / 'staging'    # renamed copies, one folder per class (rebuilt every run)
OUT   = WORK / 'merged'     # final train/val/test folders (rebuilt every run)

for p in (DRIVE, RAW):
    p.mkdir(parents=True, exist_ok=True)

DATASETS = ['defcls', 'bast']   # folder names inside RAW. defcls = dataset 1, bast = dataset 2
print('Google Drive folder :', DRIVE)
print('Working folder      :', WORK)

Mounted at /content/drive
Google Drive folder : /content/drive/MyDrive/defect_project
Working folder      : /content/work


## Step 2: Download dataset 1 from Roboflow
- You need a free Roboflow account and your **API key** (look in your Roboflow account/workspace settings for an "API key").
- When you run the cell, it asks you to paste the key into a small box. **The key is hidden as you type and is not saved in the notebook.** Never share it.
- If you see an error saying the **version** was not found, change `ROBOFLOW_VERSION` below (open the dataset page on Roboflow, click *Download Dataset*, choose *Folder Structure*, then *Show download code*, and copy the version number it shows).
- This cell downloads from the internet, so it can take a few minutes.

In [ ]:
%pip install -q roboflow
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_WORKSPACE = 'project-jkfnh'
ROBOFLOW_PROJECT   = 'defects-classification-in-3d-printing-fwsu3'
ROBOFLOW_VERSION   = 1          # <- change this if you get a "version not found" error

dest = RAW / 'defcls'
if dest.exists() and len(list(dest.rglob('*.*'))) > 10:
    print('Already downloaded:', dest)
else:
    key = getpass('Paste your Roboflow API key here and press Enter: ')
    rf = Roboflow(api_key=key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    project.version(ROBOFLOW_VERSION).download('folder', location=str(dest))
    print('Downloaded to', dest)

Paste your Roboflow API key here and press Enter: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/work/raw/defcls in folder:: 100%|██████████| 11235/11235 [00:03<00:00, 2926.56it/s]

Downloaded to /content/work/raw/defcls


In [ ]:
%pip install -q remotezip
import collections
from pathlib import Path
from remotezip import RemoteZip

BAST_URLS = {
    'base_data': 'https://zenodo.org/records/14712897/files/base_data.zip?download=1',
    'test_data': 'https://zenodo.org/records/14712897/files/test_data.zip?download=1',
}
DEPTH = 3   # how many folder levels to show

for name, url in BAST_URLS.items():
    print(f'\n=== {name} ===')
    try:
        with RemoteZip(url) as z:
            infos = [i for i in z.infolist() if not i.is_dir()]
    except Exception as e:
        print('Could not read it online:', repr(e)[:200])
        continue
    total_gb = sum(i.file_size for i in infos) / 1e9
    print(f'  files: {len(infos)}   total size when unzipped: {total_gb:.2f} GB')
    print('  file types:', dict(collections.Counter(Path(i.filename).suffix.lower() for i in infos)))
    cnt, size = collections.Counter(), collections.Counter()
    for i in infos:
        key = '/'.join(Path(i.filename).parts[:-1][:DEPTH])
        cnt[key] += 1
        size[key] += i.file_size
    for k in sorted(cnt)[:60]:
        print(f'  {cnt[k]:7d} files  {size[k]/1e9:7.2f} GB  {k}')
    if len(cnt) > 60:
        print(f'  ... and {len(cnt) - 60} more folders')
    print('  example names:', [Path(i.filename).name for i in infos[:3]])


=== base_data ===
  files: 110852   total size when unzipped: 19.66 GB
  file types: {'.jpg': 110852}
    14866 files     1.88 GB  base/cropped/extrusion_error_print
    14095 files     1.54 GB  base/cropped/layershift_error_print
    26465 files     2.90 GB  base/cropped/no_error_print
    14866 files     3.84 GB  base/full_res/extrusion_error_print
    14095 files     3.28 GB  base/full_res/layershift_error_print
    26465 files     6.22 GB  base/full_res/no_error_print
  example names: ['camera1_01.jpg', 'camera1_02.jpg', 'camera1_03.jpg']

=== test_data ===
  files: 40282   total size when unzipped: 12.63 GB
  file types: {'.jpg': 40282}
     7006 files     0.77 GB  test/cropped/extrusion_error_print
     6700 files     0.66 GB  test/cropped/layershift_error_print
     6435 files     0.63 GB  test/cropped/no_error_print
     7006 files     3.82 GB  test/full_res/extrusion_error_print
     6700 files     3.46 GB  test/full_res/layershift_error_print
     6435 files     3.30 GB  tes

## Step 3: Unzip dataset 2 (Bast et al.)
Do this **before** running the cell:
1. In your browser open `https://doi.org/10.5281/zenodo.14712897` and download the file(s) listed under **Files**.
2. Upload them to the `defect_project` folder in Google Drive (the same folder where this notebook saves its results).
3. Rename the zip to `bast.zip`. If there are several zips, name them `bast_1.zip`, `bast_2.zip`, and so on.

If the download is not a `.zip` (for example `.tar.gz` or `.7z`), stop here and tell me the file name.

In [ ]:
zips = sorted(DRIVE.glob('bast*.zip'))
if not zips:
    print('No bast zip found in', DRIVE)
    print('Upload it there and name it bast.zip (or bast_1.zip, bast_2.zip ...).')
for z in zips:
    print('Unzipping', z.name, '... (this can take a while for big files)')
    with zipfile.ZipFile(z) as f:
        f.extractall(RAW / 'bast')
if zips:
    print('Done.')

## Step 4: Look inside both datasets (this cell only reads, it changes nothing)
It prints every folder that contains images and how many images are in it. **Copy everything it prints and send it to me.**
From this I can tell you the class names and which folder level holds the classes for dataset 2.

In [ ]:
from PIL import Image

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
SPLIT_NAMES = {'train', 'val', 'valid', 'validation', 'test'}

def norm(s):
    # 'Blobs and Zits' -> 'blobs_and_zits' (the form used as keys in LABEL_MAP below)
    return re.sub(r'[^a-z0-9]+', '_', str(s).lower()).strip('_')

def summarize(root, max_lines=70):
    imgs = [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXT]
    counts = collections.Counter(str(p.parent.relative_to(root)) for p in imgs)
    print(f'  total images: {len(imgs)}')
    lines = sorted(counts.items())
    for d, n in lines[:max_lines]:
        print(f'  {n:6d}  {d}')
    if len(lines) > max_lines:
        print(f'  ... and {len(lines) - max_lines} more folders')
    print('  example file names:', [p.name for p in sorted(imgs)[:3]])

for ds in DATASETS:
    print(f'\n=== {ds} ===')
    if not (RAW / ds).exists():
        print('  folder missing (did the earlier step run?)')
        continue
    summarize(RAW / ds)


=== defcls ===
  total images: 11233
     119  test/Bed Adhesion
     104  test/Blob of Death
      95  test/Blobs and Zits
      95  test/Layer Separation
     104  test/Layer Shifting
      94  test/No Defect
     105  test/Nozzle Clog
      41  test/Over Extrusion
      93  test/Spaghetti
      95  test/Stringing
      95  test/Under Extrusion
      94  test/Warping
     864  train/Bed Adhesion
     792  train/Blob of Death
     903  train/Blobs and Zits
     800  train/Layer Separation
     867  train/Layer Shifting
     828  train/No Defect
     891  train/Nozzle Clog
     810  train/Over Extrusion
     821  train/Spaghetti
     853  train/Stringing
     831  train/Under Extrusion
     839  train/Warping
  example file names: ['-2025-07-23-150034_jpg.rf.958f44ac48524cae992db7e321b3deff.jpg', '-2025-07-23-150045_jpg.rf.2e27c7d146a3efe13b6a371ff568ccfc.jpg', '-2025-07-23-150057_jpg.rf.dc5bd7b8c39fdc7aaa3afabde72e335d.jpg']

=== bast ===
  folder missing (did the earlier step run?)


## Step 5: Decide the class names  (YOU EDIT THIS)
Here you tell the notebook how to rename the folder names into one shared list of classes.

- `LABEL_MAP['dataset']['folder_key'] = 'final_class_name'` renames a class.
- Use `None` instead of a name to **leave that class out**.
- `folder_key` is the folder name in lowercase with spaces and symbols turned into `_` (Step 4 shows the folder names; for example `Blobs and Zits` becomes `blobs_and_zits`).
- Anything you do not list is skipped, and Step 6 prints a list of what was skipped so nothing disappears silently.

The entries for `defcls` are **my suggestion** from the class list on the Roboflow page. I left out classes that are print failures rather than defects you can see on a finished part (spaghetti, nozzle clog, bed adhesion, blob of death) and the `unlabeled` class. Change them if you disagree, and check them against the Step 4 output.
The `bast` map is empty because I have not seen its folders yet. Send me the Step 4 output and I will tell you what to put.

Other settings (leave empty until I tell you):
- `CLASS_LEVEL`: if the class is not the name of the image's own folder, say which folder level it is (0 = the first folder inside the dataset folder).
- `INCLUDE_IF_PATH_CONTAINS` / `EXCLUDE_IF_PATH_CONTAINS`: for example to use only the cropped images of Bast and ignore the full-size ones, so the same photo is not counted twice.
- `GROUP_REGEX`: if file names contain a print or part ID, so all photos of one print stay in the same split.

In [ ]:
LABEL_MAP = {
    'defcls': {
        'no_defect':         'good',
        'blobs_and_zits':    'blobs_zits',
        'layer_separation':  'layer_separation',
        'layer_shifting':    'layer_shift',
        'over_extrusion':    'over_extrusion',
        'under_extrusion':   'under_extrusion',
        'stringing':         'stringing',
        'warping':           'warping',
        'z_banding':         'z_banding',
        # left out on purpose (None = skip). Change if you disagree:
        'spaghetti':         None,
        'nozzle_clog':       None,
        'bed_adhesion':      None,
        'blob_of_death':     None,
        'unlabeled':         None,
    },
    'bast': {
        # fill in after Step 4, for example:
        # 'no_defect': 'good', 'layer_shift': 'layer_shift', 'extrusion_error': 'extrusion_error',
    },
}

CLASS_LEVEL = {}                  # example: {'bast': 0}
INCLUDE_IF_PATH_CONTAINS = {}     # example: {'bast': ['crop']}
EXCLUDE_IF_PATH_CONTAINS = {}     # example: {'bast': ['full']}
GROUP_REGEX = {}                  # example: {'bast': r'^(print\d+)_'}

# Step 6 compares the datasets for look-alike images. Set to True to remove the copies found in the second dataset.
DROP_CROSS_DATASET_DUPLICATES = False

## Step 6: Build the merged dataset
This cell copies every image you kept into one place, with the new class names. It also:
- skips **exact duplicate files** (even across the two datasets),
- records a small "fingerprint" of each image, so it can warn you when the **same photo appears in both datasets** (that would make your test accuracy look better than it really is),
- keeps Roboflow's augmented copies of a photo together with the original (so they cannot end up in both train and test).

At the end it prints what it skipped and any class names it could not map. If you see a list under **UNMAPPED**, go back to Step 5, add those names, and run Step 6 again.

In [ ]:
import pandas as pd

def get_class(ds, root, p):
    rel = p.relative_to(root).parts[:-1]          # folder names between the dataset root and the image
    lvl = CLASS_LEVEL.get(ds)
    if lvl is not None:
        return rel[lvl] if len(rel) > lvl else None
    c = p.parent.name
    if norm(c) in SPLIT_NAMES:
        c = p.parent.parent.name
    return c

def safe(s):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', s)[:80]

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

RESAMPLE = Image.Resampling.LANCZOS if hasattr(Image, 'Resampling') else Image.LANCZOS

def dhash(im, size=8):
    # tiny 64-bit fingerprint: similar-looking images get the same (or nearly the same) value
    g = im.convert('L').resize((size + 1, size), RESAMPLE)
    px = list(g.getdata())
    bits = 0
    for r in range(size):
        for c in range(size):
            a, b = px[r * (size + 1) + c], px[r * (size + 1) + c + 1]
            bits = (bits << 1) | (1 if a > b else 0)
    return f'{bits:016x}'

def group_of(ds, stem):
    base = stem.split('.rf.')[0]                  # drop Roboflow's augmentation suffix
    rx = GROUP_REGEX.get(ds)
    if rx:
        m = re.match(rx, base)
        if m:
            base = m.group(1)
    return f'{ds}:{base}'

if STAGE.exists():
    shutil.rmtree(STAGE)
rows, seen_md5 = [], set()
skipped, unmapped = collections.Counter(), collections.Counter()

for ds in DATASETS:
    root = RAW / ds
    if not root.exists():
        print(f'[{ds}] folder missing, skipped')
        continue
    cmap = LABEL_MAP.get(ds, {})
    inc = [s.lower() for s in INCLUDE_IF_PATH_CONTAINS.get(ds, [])]
    exc = [s.lower() for s in EXCLUDE_IF_PATH_CONTAINS.get(ds, [])]
    for p in sorted(root.rglob('*')):
        if p.suffix.lower() not in IMG_EXT:
            continue
        rel = str(p.relative_to(root)).lower()
        if (inc and not any(s in rel for s in inc)) or any(s in rel for s in exc):
            skipped['path_filter'] += 1
            continue
        src = get_class(ds, root, p)
        if src is None:
            skipped['no_class_folder'] += 1
            continue
        tgt = cmap.get(norm(src), '__unmapped__')
        if tgt == '__unmapped__':
            unmapped[(ds, src)] += 1
            continue
        if tgt is None:
            skipped['dropped_class'] += 1
            continue
        h = md5(p)
        if h in seen_md5:
            skipped['exact_duplicate'] += 1
            continue
        seen_md5.add(h)
        try:
            with Image.open(p) as im:
                dh = dhash(im)
        except Exception:
            skipped['unreadable_image'] += 1
            continue
        out = STAGE / tgt / f'{ds}__{safe(p.stem)}__{h[:6]}{p.suffix.lower()}'
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, out)
        rows.append(dict(path=str(out), cls=tgt, dataset=ds, group=group_of(ds, p.stem), dhash=dh))

df = pd.DataFrame(rows)
print(f'Kept {len(df)} images.  Skipped: {dict(skipped)}')
if unmapped:
    print('\nUNMAPPED (not in LABEL_MAP, so skipped). Add each one to LABEL_MAP, or map it to None to leave it out on purpose:')
    for (ds, src), n in unmapped.items():
        print(f'  {ds}: {src!r}  (key {norm(src)!r})  x{n}')
assert len(df) > 0, 'No images were kept. Check LABEL_MAP and the Step 4 output.'

# look-alike images that appear in more than one dataset
order = {d: i for i, d in enumerate(DATASETS)}
df['_o'] = df['dataset'].map(order)
per_hash = df.groupby('dhash')['dataset'].nunique()
cross = per_hash[per_hash > 1].index
print(f'\nImages with a look-alike in another dataset: {int(df["dhash"].isin(cross).sum())} '
      f'({len(cross)} fingerprints). Set DROP_CROSS_DATASET_DUPLICATES = True in Step 5 if this is a large share.')
if DROP_CROSS_DATASET_DUPLICATES and len(cross):
    first = df[df['dhash'].isin(cross)].groupby('dhash')['_o'].min()
    drop = df['dhash'].isin(cross) & (df['_o'] > df['dhash'].map(first))
    print(f'Removing {int(drop.sum())} look-alike images from the later dataset.')
    df = df[~drop].reset_index(drop=True)
df = df.drop(columns=['_o'])

## Step 7: Split into train / val / test and save to Drive
- **train** (about 72 %): the model learns from these.
- **val** (about 14 %): used while training to compare settings.
- **test** (about 14 %): used once at the end to report accuracy.
Photos from the same group (same original photo and its augmented copies) always go to the same split. Each class is spread over the three splits as evenly as possible.

The cell prints a table of images per class and warns you about classes that are too small. It then saves:
- `merged_dataset.zip` in your Drive (the folders train/val/test with one subfolder per class),
- `merged_split.csv` in your Drive (which image went where, and from which dataset, so we can check accuracy per dataset later).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
folds = [te for _, te in sgkf.split(df, df['cls'], df['group'])]
df['split'] = 'train'
df.loc[folds[0], 'split'] = 'test'     # about 14 %
df.loc[folds[1], 'split'] = 'val'      # about 14 %

if OUT.exists():
    shutil.rmtree(OUT)
for r in df.itertuples():
    dst = OUT / r.split / r.cls / Path(r.path).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(r.path, dst)

table = df.pivot_table(index='cls', columns='split', values='path', aggfunc='count', fill_value=0)
table = table.reindex(columns=['train', 'val', 'test'], fill_value=0)
table['total'] = table.sum(axis=1)
print(table)
print()
print('Images per dataset:')
print(df.groupby(['dataset', 'cls']).size().unstack(fill_value=0).T)

print()
for c, row in table.iterrows():
    if row['total'] < 50:
        print(f'WARNING: class {c!r} has only {row["total"]} images. Consider merging it or leaving it out.')
    if row['val'] == 0 or row['test'] == 0:
        print(f'WARNING: class {c!r} is missing from val or test, so its accuracy cannot be measured.')

df.drop(columns=['path', 'dhash']).assign(file=df['path'].map(lambda s: Path(s).name)).to_csv(DRIVE / 'merged_split.csv', index=False)
print('\nZipping the merged dataset to Drive (can take a few minutes)...')
shutil.make_archive(str(DRIVE / 'merged_dataset'), 'zip', OUT)
print('Saved:', DRIVE / 'merged_dataset.zip')

## Step 8: Folders for your own phone photos (accuracy check only)
This makes an empty folder for each class inside `defect_project/phone_test` in Drive. Later, put your phone photos of printed parts in the matching folders. They are **never used for training**, only to check the finished model.

In [ ]:
PHONE = DRIVE / 'phone_test'
for c in sorted(df['cls'].unique()):
    (PHONE / c).mkdir(parents=True, exist_ok=True)
print('Created:', PHONE)
print('Classes:', sorted(df['cls'].unique()))